# Where do three chromosome regions meet?
Harju, Messelink and Broedersz used simulated bacterial chromosomes to ask which models of chromosome organization are compatible with three-point contacts. Here you can select a bait locus and see where the loop-extruder model predicts the other two loci will meet it. [Paper](https://doi.org/10.1103/PhysRevE.111.014403)

Run the cells in order. The opening view uses the original 404-bin condensin map, a radius of five monomer lengths and the authors’ bait at bin 200. The stored Julia plane is checked against an independent NumPy calculation before display.

In [ ]:
import json
from pathlib import Path
import subprocess
import sys
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/worker.jl").is_file())
sys.path.insert(0, str(ROOT / "notebooks"))
from figures import heatmaps, load_run, predict_slice

runs = sorted(p.parent for p in (ROOT / "results").glob("*/run.json"))
sources = [ROOT / "examples/published-slice", ROOT / "examples/laptop", *runs]
picker = widgets.Dropdown(options=[(p.name, p) for p in sources], description="Dataset")
display(picker)

Select a dataset above and run the next cell. Move the slider to compare the simulated pair-contact frequencies (left) with predicted triplet probabilities (right). Each pixel on the right represents a triple with the selected bait. Brighter colors indicate larger probabilities. The scales saturate at the upper colorbar limits. The circular model excludes repeated and immediately adjacent bins.

In [ ]:
run, contacts, saved = load_run(picker.value)
def show_bait(bait):
    plane = saved if bait == run["bait"] else predict_slice(contacts, bait)
    fig = heatmaps(contacts, plane, bait, run["spacing_kb"])
    display(fig)
    plt.close(fig)
slider = widgets.IntSlider(min=1, max=run["n"], value=run["bait"],
                           description="Bait bin", continuous_update=False)
display(widgets.interactive_output(show_bait, {"bait": slider}), slider)

## What changes when ADA adds cores?
Select one completed run. The included laptop measurement uses 101 bins. The ADA scaling job uses 404. Each series compares the same input and hardware. Speedup uses the median one-thread time of the threaded implementation. Efficiency divides speedup by the thread count.

In [ ]:
measured = [ROOT / "examples/laptop"] + [p for p in runs if json.loads((p/"run.json").read_text())["summary"]]
perf = widgets.Dropdown(options=[(p.name, p) for p in measured], value=measured[-1], description="Benchmark")
def performance(folder):
    data = json.loads((folder / "run.json").read_text())
    print(f"{data['n']} bins · {data['cpu']} · Julia {data['julia']} · job {data['job'] or 'local'}")
    print(f"Output {data['tensor_bytes']/2**20:.2f} MiB · requested {data['requested_mem_mb'] or '—'} MiB · {data['validation']}")
    print("Method      Threads    Median (s)   Speedup   Efficiency   Avg cores   Peak RSS (MiB)")
    for row in data["summary"]:
        s = f"{row['speedup']:.2f}×" if row['speedup'] else "—"
        e = f"{row['efficiency']:.0%}" if row['efficiency'] else "—"
        print(f"{row['mode']:10} {row['threads']:7} {row['median_s']:12.6f} {s:>9} {e:>12} {row['cpu_cores']:11.2f} {row['peak_rss_mib']:15.1f}")
    rows = [row for row in data["summary"] if row["speedup"] is not None]
    if len(rows) >= 2:
        x = [r["threads"] for r in rows]
        fig, ax = plt.subplots(figsize=(6, 3), layout="constrained")
        ax.plot(x, x, "--", color="gray", label="Ideal scaling")
        ax.plot(x, [r["speedup"] for r in rows], "o-", label="Measured")
        ax.set(xlabel="Julia threads", ylabel="Speedup", xticks=x)
        ax.legend()
        display(fig)
        plt.close(fig)
display(perf, widgets.interactive_output(performance, {"folder": perf}))

## Run a calculation
The first command runs the laptop-sized validation, including three small timings per configuration. Uncomment it to run. Rerun the first notebook cell to discover the new results.

For the full calculation, set your ADA partition in the batch cell and uncomment the submission line. The Julia executable and Python 3.11+ must be visible in the batch environment. If Slurm commands are unavailable inside your OOD app, use the same commands in the portal shell.

In [ ]:
# subprocess.run([sys.executable, "demo.py", "smoke"], cwd=ROOT, check=True)

In [ ]:
def submit_scaling(partition, cpus=8):
    command = ["sbatch", "--parsable", f"--partition={partition}", f"--cpus-per-task={cpus}", "slurm/scale.sbatch"]
    return subprocess.check_output(command, cwd=ROOT, text=True).strip().split(";")[0]

# job = submit_scaling("YOUR_DEPARTMENT_PARTITION")
# subprocess.run(["squeue", "-j", job])
# subprocess.run(["sacct", "-j", job, "--format=JobID,State,Submit,Start,Elapsed,MaxRSS"])